# GAT Autoencoder â€” MPLS Network Anomaly Detection

**Goal**: Train a Graph Attention Network (GAT) autoencoder to detect network anomalies
from graph-structured telemetry snapshots. Reconstruction error per node = anomaly score.

**Data**: `graph_snapshots_train.pkl` â€” ~2160 graph snapshots over 3 days, one every 2 minutes.
Each snapshot: node_features (10, 20), node_labels (10,), edge_index (COO).

**Architecture**:
  GATEncoder(20â†’128Ã—8 heads â†’64Ã—4 heads â†’32) â†’ GATDecoder(32â†’128â†’20)

**Topology**: 10 nodes (3 CE, 2 PE, 3 P, 2 DC-CE) with 12 edges in COO format.

In [ ]:
# Cell 0: Install dependencies (run once)
!pip install -q torch-geometric optuna
print("Dependencies installed.")

In [ ]:
# Cell 1: Imports
import os, sys, json, gc, pickle, warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

import torch_geometric
from torch_geometric.nn import GATConv
from torch_geometric.data import Data, Batch

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             precision_recall_curve, roc_curve,
                             confusion_matrix, classification_report)

import optuna

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
print(f"PyTorch {torch.__version__}, torch_geometric {torch_geometric.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}, devices: {torch.cuda.device_count()}")
# Twin GPU note: GAT model is small (10 nodes, batch=256) â€” single GPU is sufficient.
# For multi-GPU, use: `from torch_geometric.nn import DataParallel` and wrap the model.
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
# Cell 2: Configuration
class Config:
    data_dir = '/kaggle/input/mpls-telemetry'
    train_pkl = os.path.join(data_dir, 'graph_snapshots_train.pkl')
    val_pkl = os.path.join(data_dir, 'graph_snapshots_val.pkl')
    
    # Graph parameters
    n_nodes = 10
    n_features = 20   # first 20 NUMERIC_COLS for GAT
    
    # Model hyperparams (will be tuned)
    latent_dim = 32
    heads_1 = 8
    heads_2 = 4
    hidden_1 = 128
    hidden_2 = 64
    dropout = 0.2
    
    # Training
    batch_size = 256
    max_epochs = 150
    learning_rate = 1e-3
    weight_decay = 1e-5
    
    # Optuna
    n_trials = 25
    
    # Output
    model_dir = '/kaggle/working'
    
    # Node ID list (for interpretability) â€” must match NODE_IDS order from topology.py
    node_ids = ['PE-1', 'PE-2', 'P-1', 'P-2', 'P-3',
                'CE-B1', 'CE-B2', 'CE-B3', 'CE-DC1', 'CE-DC2']
    
cfg = Config()

In [ ]:
# Cell 3: Data Loading
print("Loading graph snapshots...")

with open(cfg.train_pkl, 'rb') as f:
    train_snapshots = pickle.load(f)
print(f"Training snapshots: {len(train_snapshots)}")

with open(cfg.val_pkl, 'rb') as f:
    val_snapshots = pickle.load(f)
print(f"Validation snapshots: {len(val_snapshots)}")

# Inspect one snapshot
s = train_snapshots[0]
print(f"\nSnapshot keys: {list(s.keys())}")
print(f"node_features shape: {s['node_features'].shape}")
print(f"node_labels shape: {s['node_labels'].shape}")
print(f"edge_index: {len(s['edge_index'][0])} edges")
print(f"timestamp: {s['timestamp']}")
print(f"\nAnomaly distribution in training:")
train_labels = np.concatenate([s['node_labels'] for s in train_snapshots])
print(f"  Normal: {(train_labels == 0).sum()}, Anomaly: {(train_labels == 1).sum()}")
print(f"  Anomaly rate: {train_labels.mean():.3%}")

In [ ]:
# Cell 4: Exploratory Data Analysis

# 4a â€” Extract per-snapshot anomaly counts
anom_counts = [s['node_labels'].sum() for s in train_snapshots]

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# 4a: Anomalous nodes per snapshot
axes[0, 0].hist(anom_counts, bins=range(0, 12), align='left', rwidth=0.8)
axes[0, 0].set_title('Anomalous Nodes per Snapshot (Train)')
axes[0, 0].set_xlabel('Count of anomalous nodes')
axes[0, 0].set_ylabel('Frequency')

# 4b: Per-node anomaly prevalence
node_anom_counts = {nid: [] for nid in cfg.node_ids}
for s in train_snapshots:
    for i, nid in enumerate(cfg.node_ids):
        node_anom_counts[nid].append(s['node_labels'][i])
node_anom_rates = {nid: np.mean(vals) for nid, vals in node_anom_counts.items()}
sorted_nodes = sorted(node_anom_rates.items(), key=lambda x: x[1])
axes[0, 1].barh([n for n, _ in sorted_nodes], [v for _, v in sorted_nodes])
axes[0, 1].set_title('Per-Node Anomaly Rate (Train)')
axes[0, 1].set_xlabel('Anomaly Rate')

# 4c â€” fill axes[0,2] with a fault phase breakdown
phases = [s['node_labels'].sum() > 0 for s in train_snapshots]
phases_str = ['Normal'] * len(train_snapshots)
# Check which snapshots are in fault phases by checking first node's phase
# (simplified: count total snapshots with any anomaly vs without)
anom_count = sum(phases)
normal_count = len(phases) - anom_count
axes[0, 2].bar(['Normal', 'Anomalous'], [normal_count, anom_count])
axes[0, 2].set_title('Snapshot Classification (Train)')
axes[0, 2].set_ylabel('Count')

# 4d: Feature distributions â€” separate figure, 2x2 grid for 4 features
feature_names = [
    'bytes_in', 'bytes_out', 'errors_in', 'drops_in',
    'drops_out', 'utilization_pct', 'bgp_sessions_active', 'bgp_prefixes_received',
    'bgp_updates_per_min', 'bgp_withdrawals_per_min', 'ospf_spf_runs',
    'ldp_sessions_active', 'mpls_lsp_count', 'mpls_label_table_size', 'vpn_routes_count',
    'cpu_load_pct', 'memory_used_pct', 'queue_depth', 'latency_ms', 'jitter_ms',
]
fig2, axes2 = plt.subplots(2, 2, figsize=(12, 8))
for i in range(4):
    ax = axes2[i // 2, i % 2]
    ax.hist(all_features[:, i], bins=80, alpha=0.7)
    ax.set_title(feature_names[i] if i < len(feature_names) else f'Feature {i}')
plt.tight_layout()
plt.show()

# 4e: Anomaly rate over time (snapshot index)
plt.figure(figsize=(14, 4))
snapshot_anom_rates = [s['node_labels'].mean() for s in train_snapshots]
plt.plot(snapshot_anom_rates, linewidth=0.6, alpha=0.7)
plt.axhline(y=np.mean(snapshot_anom_rates), color='red', linestyle='--',
            label=f'Mean={np.mean(snapshot_anom_rates):.3%}')
plt.xlabel('Snapshot Index')
plt.ylabel('Anomaly Rate')
plt.title('Anomaly Rate Over Time (Train Snapshots)')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Cell 5: Feature Engineering â€” Normalization & PyG Dataset

# Edge index is static across all snapshots (network topology doesn't change)
edge_index = train_snapshots[0]['edge_index']
edge_tensor = torch.tensor(edge_index, dtype=torch.long)
print(f"Edge index: {len(edge_index[0])} edges, shape={edge_tensor.shape}")

# Build feature matrix: (n_snapshots * n_nodes, n_features)
X_all = np.vstack([s['node_features'] for s in train_snapshots])
y_train_nodes = np.concatenate([s['node_labels'] for s in train_snapshots])
print(f"All training features shape: {X_all.shape}")

# Normalize per-feature — fit scaler on NORMAL rows only
# anomalous values become reconstruction outliers; transform all data.
normal_mask = np.concatenate([s['node_labels'] for s in train_snapshots]) == 0
y_train_nodes = np.concatenate([s['node_labels'] for s in train_snapshots])
normal_mask = y_train_nodes == 0
scaler = StandardScaler()
scaler.fit(X_all[normal_mask])
print(f'Scaler fitted on {normal_mask.sum():,} normal rows, ignored {(~normal_mask).sum():,} anomalous')
scaler.fit(X_all[normal_mask])
X_norm = scaler.transform(X_all).astype(np.float32)
print(f"Scaler fitted on {normal_mask.sum():,} normal rows (ignored {(~normal_mask).sum():,} anomalous)")

# Reshape back: (n_snapshots, n_nodes, n_features)
X_train_norm = X_norm.reshape(len(train_snapshots), cfg.n_nodes, cfg.n_features)
y_train = np.array([s['node_labels'] for s in train_snapshots])
print(f"X_train_norm: {X_train_norm.shape}, y_train: {y_train.shape}")
print(f"Normalized: mean~{X_train_norm.mean():.4f}, std~{X_train_norm.std():.4f}")

# Validation
X_val_all = np.vstack([s['node_features'] for s in val_snapshots])
X_val_norm = scaler.transform(X_val_all).astype(np.float32)
X_val_norm = X_val_norm.reshape(len(val_snapshots), cfg.n_nodes, cfg.n_features)
y_val = np.array([s['node_labels'] for s in val_snapshots])
print(f"X_val_norm: {X_val_norm.shape}, y_val: {y_val.shape}")

# Save scaler
import joblib
scaler_path = os.path.join(cfg.model_dir, 'gat_scaler.pkl')
joblib.dump(scaler, scaler_path)
print(f"Scaler saved to {scaler_path}")

In [ ]:
# Cell 6: PyG Dataset & Model Definition

class GraphSnapshotDataset(Dataset):
    """Dataset that returns a single PyG Data object per snapshot."""
    def __init__(self, features, labels, edge_index):
        """
        features: (n_snapshots, n_nodes, n_features)
        labels: (n_snapshots, n_nodes)
        edge_index: tuple (src_list, dst_list)
        """
        self.n_snapshots = len(features)
        self.graphs = []
        for i in range(self.n_snapshots):
            x = torch.tensor(features[i], dtype=torch.float)
            y = torch.tensor(labels[i], dtype=torch.long)
            self.graphs.append(data)
    
    def __len__(self):
        return self.n_snapshots
    
    def __getitem__(self, idx):
        return self.graphs[idx]


class GATEncoder(nn.Module):
    def __init__(self, n_features, latent_dim, hidden_1=128, hidden_2=64,
                 heads_1=8, heads_2=4, dropout=0.2):
        super().__init__()
        self.conv1 = GATConv(n_features, hidden_1, heads=heads_1, dropout=dropout)
        self.conv2 = GATConv(hidden_1 * heads_1, hidden_2, heads=heads_2, dropout=dropout)
        self.conv3 = GATConv(hidden_2 * heads_2, latent_dim, heads=1)
        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()
    
    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        x = self.dropout(x)
        x = self.conv2(x, edge_index).relu()
        x = self.dropout(x)
        return self.conv3(x, edge_index)


class GATDecoder(nn.Module):
    def __init__(self, n_features, latent_dim):
        super().__init__()
        self.fc1 = nn.Linear(latent_dim, 128)
        self.fc2 = nn.Linear(128, n_features)
    
    def forward(self, z):
        return self.fc2(self.fc1(z).relu())


class GATAutoencoder(nn.Module):
    def __init__(self, n_features=20, latent_dim=32, hidden_1=128, hidden_2=64,
                 heads_1=8, heads_2=4, dropout=0.2):
        super().__init__()
        self.encoder = GATEncoder(n_features, latent_dim, hidden_1, hidden_2,
                                   heads_1, heads_2, dropout)
        self.decoder = GATDecoder(n_features, latent_dim)
    
    def forward(self, x, edge_index):
        z = self.encoder(x, edge_index)
        return self.decoder(z)
    
    def reconstruction_error(self, x, edge_index):
        x_hat = self.forward(x, edge_index)
        return torch.mean((x - x_hat) ** 2, dim=1)


train_dataset = GraphSnapshotDataset(X_train_norm, y_train, edge_index)
val_dataset = GraphSnapshotDataset(X_val_norm, y_val, edge_index)
print(f"Train dataset: {len(train_dataset)} graphs")
print(f"Val dataset:   {len(val_dataset)} graphs")

In [ ]:
# Cell 7: Training Utilities

def collate_graphs(batch):
    """Collate list of PyG Data objects into a Batch."""
    return Batch.from_data_list(batch)


def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    total_nodes = 0
    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        x_hat = model(batch.x, batch.edge_index)
        loss = criterion(x_hat, batch.x)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item() * batch.x.size(0)
        total_nodes += batch.x.size(0)
    return total_loss / total_nodes


@torch.no_grad()
def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    total_nodes = 0
    all_errors = []
    all_labels = []
    for batch in loader:
        batch = batch.to(device)
        x_hat = model(batch.x, batch.edge_index)
        loss = criterion(x_hat, batch.x)
        err = model.reconstruction_error(batch.x, batch.edge_index)
        total_loss += loss.item() * batch.x.size(0)
        total_nodes += batch.x.size(0)
        all_errors.append(err.cpu())
        if hasattr(batch, 'y'):
            all_labels.append(batch.y.cpu())
    errors = torch.cat(all_errors).numpy()
    labels = torch.cat(all_labels).numpy() if all_labels else None
    return total_loss / total_nodes, errors, labels


def find_best_threshold(errors_normal, errors_anom):
    all_errors = np.concatenate([errors_normal, errors_anom])
    all_labels = np.concatenate([np.zeros_like(errors_normal), np.ones_like(errors_anom)])
    precisions, recalls, thresholds = precision_recall_curve(all_labels, all_errors)
    f1_scores = 2 * precisions[:-1] * recalls[:-1] / (precisions[:-1] + recalls[:-1] + 1e-10)
    best_idx = np.argmax(f1_scores)
    return thresholds[best_idx], f1_scores[best_idx], all_errors, all_labels

In [ ]:
# Cell 8: Hyperparameter Tuning with Optuna

# Use subset of validation for tuning
tune_size = min(500, len(val_dataset))
val_indices = np.random.RandomState(42).choice(len(val_dataset), tune_size, replace=False)
val_subset = [val_dataset[i] for i in val_indices]
val_loader_tune = DataLoader(val_subset, batch_size=cfg.batch_size,
                              collate_fn=collate_graphs, shuffle=False)

def objective(trial):
    # Suggest hyperparameters
    latent_dim = trial.suggest_categorical('latent_dim', [16, 32, 64])
    hidden_1 = trial.suggest_categorical('hidden_1', [64, 128, 256])
    hidden_2 = trial.suggest_categorical('hidden_2', [32, 64, 128])
    heads_1 = trial.suggest_categorical('heads_1', [4, 8])
    heads_2 = trial.suggest_categorical('heads_2', [2, 4])
    lr = trial.suggest_float('lr', 3e-4, 3e-3, log=True)
    dropout = trial.suggest_float('dropout', 0.1, 0.4)
    weight_decay = trial.suggest_float('weight_decay', 1e-6, 1e-4, log=True)
    
    model = GATAutoencoder(
        n_features=cfg.n_features,
        latent_dim=latent_dim,
        hidden_1=hidden_1,
        hidden_2=hidden_2,
        heads_1=heads_1,
        heads_2=heads_2,
        dropout=dropout,
    ).to(device)
    
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.MSELoss()
    
    # Subset of training data
    train_size = min(1000, len(train_dataset))
    train_idx = np.random.RandomState(42).choice(len(train_dataset), train_size, replace=False)
    train_subset = [train_dataset[i] for i in train_idx]
    train_loader = DataLoader(train_subset, batch_size=cfg.batch_size,
                              collate_fn=collate_graphs, shuffle=True)
    
    # Train
    best_val_loss = float('inf')
    patience = 5
    patience_counter = 0
    
    for epoch in range(40):
        train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
        val_loss, _, _ = eval_epoch(model, val_loader_tune, criterion, device)
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break
        
        trial.report(val_loss, epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()
    
    return best_val_loss


print("Starting Optuna hyperparameter search...")
print(f"Trials: {cfg.n_trials}")
study = optuna.create_study(
    direction='minimize',
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(),
)
study.optimize(objective, n_trials=cfg.n_trials, show_progress_bar=True)

print(f"\nBest trial: {study.best_trial.number}")
print(f"Best val loss: {study.best_value:.6f}")
print(f"Best params: {study.best_params}")

best_params = study.best_params
cfg.latent_dim = best_params['latent_dim']
cfg.hidden_1 = best_params['hidden_1']
cfg.hidden_2 = best_params['hidden_2']
cfg.heads_1 = best_params['heads_1']
cfg.heads_2 = best_params['heads_2']
cfg.learning_rate = best_params['lr']
cfg.dropout = best_params['dropout']
cfg.weight_decay = best_params['weight_decay']

In [ ]:
# Cell 9: Optuna Visualization
fig = optuna.visualization.plot_optimization_history(study)
fig.update_layout(width=900, height=400)
fig.show()

fig2 = optuna.visualization.plot_param_importances(study)
fig2.update_layout(width=900, height=500)
fig2.show()

fig3 = optuna.visualization.plot_parallel_coordinate(study)
fig3.update_layout(width=900, height=500)
fig3.show()

In [ ]:
# Cell 10: Final Training with Best Hyperparameters

print("Building final model with best hyperparameters...")
print(f"  latent_dim={cfg.latent_dim}, hidden_1={cfg.hidden_1}, hidden_2={cfg.hidden_2}")
print(f"  heads_1={cfg.heads_1}, heads_2={cfg.heads_2}, lr={cfg.learning_rate:.6f}")
print(f"  dropout={cfg.dropout}, weight_decay={cfg.weight_decay:.6f}")

model = GATAutoencoder(
    n_features=cfg.n_features,
    latent_dim=cfg.latent_dim,
    hidden_1=cfg.hidden_1,
    hidden_2=cfg.hidden_2,
    heads_1=cfg.heads_1,
    heads_2=cfg.heads_2,
    dropout=cfg.dropout,
).to(device)

optimizer = optim.AdamW(model.parameters(), lr=cfg.learning_rate, weight_decay=cfg.weight_decay)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=8)
criterion = nn.MSELoss()

train_loader = DataLoader(train_dataset, batch_size=cfg.batch_size,
                           collate_fn=collate_graphs, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=cfg.batch_size,
                         collate_fn=collate_graphs, shuffle=False)

print(f"Train: {len(train_loader)} batches, Val: {len(val_loader)} batches")

history = {'train_loss': [], 'val_loss': []}
best_val_loss = float('inf')
patience = 15
patience_counter = 0

for epoch in range(1, cfg.max_epochs + 1):
    train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, _, _ = eval_epoch(model, val_loader, criterion, device)
    scheduler.step(val_loss)
    
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({
            'model_state': model.state_dict(),
            'model_params': {
                'n_features': cfg.n_features,
                'latent_dim': cfg.latent_dim,
                'hidden_1': cfg.hidden_1,
                'hidden_2': cfg.hidden_2,
                'heads_1': cfg.heads_1,
                'heads_2': cfg.heads_2,
                'dropout': cfg.dropout,
            },
            'optimizer_state': optimizer.state_dict(),
            'epoch': epoch,
            'val_loss': val_loss,
        }, os.path.join(cfg.model_dir, 'gat_best.pt'))
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch}")
            break
    
    if epoch % 10 == 0 or epoch == 1:
        lr_now = optimizer.param_groups[0]['lr']
        print(f"Epoch {epoch:3d} | train_loss={train_loss:.6f} | val_loss={val_loss:.6f} | lr={lr_now:.2e}")

print(f"\nBest val_loss: {best_val_loss:.6f}")

In [ ]:
# Cell 11: Training History Plot
plt.figure(figsize=(10, 5))
plt.plot(history['train_loss'], label='Train Loss', alpha=0.8)
plt.plot(history['val_loss'], label='Val Loss', alpha=0.8)
plt.axvline(x=np.argmin(history['val_loss']), color='green', linestyle='--', alpha=0.5, label='Best Model')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('GAT Training History')
plt.legend()
plt.yscale('log')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Cell 12: Threshold Calibration

checkpoint = torch.load(os.path.join(cfg.model_dir, 'gat_best.pt'), map_location=device)
model.load_state_dict(checkpoint['model_state'])
model.eval()
criterion = nn.MSELoss()
print(f"Loaded best model from epoch {checkpoint['epoch']}, val_loss={checkpoint['val_loss']:.6f}")

# Full validation evaluation
# We compute per-NODE reconstruction errors (flattened across all snapshots)
_, val_errors, val_labels_flat = eval_epoch(model, val_loader, criterion, device)

print(f"Val errors: {val_errors.shape}, labels: {val_labels_flat.shape}")

normal_mask = val_labels_flat == 0
anom_mask = val_labels_flat == 1
val_errors_normal = val_errors[normal_mask]
val_errors_anom = val_errors[anom_mask]

print(f"Normal nodes: {len(val_errors_normal)}, mean_err={val_errors_normal.mean():.6f}")
print(f"Anomalous nodes: {len(val_errors_anom)}, mean_err={val_errors_anom.mean():.6f}")

# Find best threshold
best_thresh, best_f1, all_errors, all_labels = find_best_threshold(
    val_errors_normal, val_errors_anom
)
print(f"\nBest threshold: {best_thresh:.6f} (F1={best_f1:.4f})")

# AUROC and AP
auroc = roc_auc_score(all_labels, all_errors)
ap = average_precision_score(all_labels, all_errors)
print(f"AUROC: {auroc:.4f}")
print(f"Avg Precision: {ap:.4f}")

In [ ]:
# Cell 13: Evaluation Plots

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# â€” 13a: Reconstruction error distributions
ax = axes[0, 0]
ax.hist(val_errors_normal, bins=100, alpha=0.6,
        label=f'Normal (n={len(val_errors_normal):,})', density=True)
ax.hist(val_errors_anom, bins=100, alpha=0.6,
        label=f'Anomaly (n={len(val_errors_anom):,})', density=True)
ax.axvline(best_thresh, color='red', linestyle='--', label=f'Threshold={best_thresh:.4f}')
ax.set_xlabel('Reconstruction Error')
ax.set_ylabel('Density')
ax.set_title('Per-Node Reconstruction Error Distribution')
ax.legend()

# â€” 13b: ROC Curve
ax = axes[0, 1]
fpr, tpr, _ = roc_curve(all_labels, all_errors)
ax.plot(fpr, tpr, lw=2, label=f'AUROC = {auroc:.4f}')
ax.plot([0, 1], [0, 1], 'k--', alpha=0.5)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve (Per-Node)')
ax.legend()

# â€” 13c: Precision-Recall Curve
ax = axes[1, 0]
precisions, recalls, _ = precision_recall_curve(all_labels, all_errors)
ax.plot(recalls, precisions, lw=2, label=f'AP = {ap:.4f}')
ax.axhline(y=all_labels.mean(), color='gray', linestyle='--', alpha=0.5, label='Baseline')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curve (Per-Node)')
ax.legend()

# â€” 13d: Confusion Matrix
ax = axes[1, 1]
preds = (all_errors >= best_thresh).astype(int)
cm = confusion_matrix(all_labels, preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Normal', 'Anomaly'],
            yticklabels=['Normal', 'Anomaly'])
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title(f'Confusion Matrix (thresh={best_thresh:.4f})')

plt.tight_layout()
plt.show()

print("\nClassification Report:")
print(classification_report(all_labels, preds, target_names=['Normal', 'Anomaly']))

In [ ]:
# Cell 14: Per-Node Performance

# Reshape: (n_snapshots, n_nodes) â†’ (n_snapshots * n_nodes,)
y_val_flat = y_val.ravel()

node_results = []
for i, nid in enumerate(cfg.node_ids):
    node_mask = np.arange(len(y_val_flat)) % cfg.n_nodes == i
    node_errs = val_errors[node_mask]
    node_labels = y_val_flat[node_mask]
    if len(np.unique(node_labels)) < 2:
        continue
    auroc_node = roc_auc_score(node_labels, node_errs)
    ap_node = average_precision_score(node_labels, node_errs)
    node_preds = (node_errs >= best_thresh).astype(int)
    f1 = 2 * (node_preds * node_labels).sum() / max((node_preds + node_labels).sum(), 1)
    node_results.append({'node_id': nid, 'auroc': auroc_node, 'ap': ap_node, 'f1': f1,
                         'n_normal': (node_labels == 0).sum(), 'n_anom': (node_labels == 1).sum()})
    print(f"  {nid}: AUROC={auroc_node:.4f}, AP={ap_node:.4f}, F1={f1:.4f}, "
          f"n={len(node_labels):,} (anom={node_labels.sum():,})")

if node_results:
    node_df = pd.DataFrame(node_results).set_index('node_id')
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    for i, col in enumerate(['auroc', 'ap', 'f1']):
        colors = ['red' if v < 0.7 else 'orange' if v < 0.85 else 'green' for v in node_df[col]]
        axes[i].bar(node_df.index, node_df[col], color=colors)
        axes[i].set_title(f'Per-Node {col.upper()}')
        axes[i].tick_params(axis='x', rotation=45)
        axes[i].set_ylim(0, 1.05)
        axes[i].axhline(y=node_df[col].mean(), color='gray', linestyle='--', alpha=0.7,
                        label=f'Mean={node_df[col].mean():.3f}')
        axes[i].legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# Cell 15: Network-Level Scoring Analysis

# Compute network-level anomaly score per snapshot (max per-node error)
val_errors_2d = val_errors.reshape(len(val_dataset), cfg.n_nodes)
network_scores = val_errors_2d.max(axis=1)

# Any anomaly in snapshot?
y_network = y_val.max(axis=1)

auroc_net = roc_auc_score(y_network, network_scores)
ap_net = average_precision_score(y_network, network_scores)
print(f"Network-Level Detection:")
print(f"  AUROC: {auroc_net:.4f}")
print(f"  Avg Precision: {ap_net:.4f}")

# Per-snapshot detection rate
net_preds = (network_scores >= best_thresh).astype(int)
correct = (net_preds == y_network).sum()
print(f"  Snapshot accuracy: {correct}/{len(y_network)} = {correct / len(y_network):.2%}")

# Plot network scores over time for val set
plt.figure(figsize=(14, 5))
colors = ['red' if a else 'blue' for a in y_network]
plt.scatter(range(len(network_scores)), network_scores, c=colors, alpha=0.5, s=10)
plt.axhline(best_thresh, color='green', linestyle='--', label=f'Threshold={best_thresh:.4f}')
plt.xlabel('Snapshot Index')
plt.ylabel('Network Score (Max Node Error)')
plt.title('Network-Level Anomaly Scores Over Time (Validation)')
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='blue', label='Normal Snapshot', alpha=0.7),
                   Patch(facecolor='red', label='Anomalous Snapshot', alpha=0.7)]
plt.legend(handles=legend_elements)
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Cell 16: Error Propagation Across Topology

# For each snapshot, compute per-node error and visualize which nodes are flagged
print("Propagation Analysis:\n")
print("Node-level confusion matrix on validation set:\n")

preds_2d = (val_errors_2d >= best_thresh).astype(int)
for i, nid in enumerate(cfg.node_ids):
    tp = (preds_2d[:, i] * y_val[:, i]).sum()
    fp = (preds_2d[:, i] * (1 - y_val[:, i])).sum()
    fn = ((1 - preds_2d[:, i]) * y_val[:, i]).sum()
    tn = ((1 - preds_2d[:, i]) * (1 - y_val[:, i])).sum()
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-10)
    print(f"  {nid:10s}: TP={tp:4.0f} FP={fp:4.0f} FN={fn:4.0f} TN={tn:4.0f}  "
          f"P={precision:.3f} R={recall:.3f} F1={f1:.3f}")

# Visualise error on a single anomalous snapshot
anom_snapshot_idx = np.where(y_network == 1)[0][0]
snapshot_errors = val_errors_2d[anom_snapshot_idx]

plt.figure(figsize=(10, 5))
bar_colors = ['red' if y_val[anom_snapshot_idx, i] else 'steelblue' for i in range(cfg.n_nodes)]
plt.bar(cfg.node_ids, snapshot_errors, color=bar_colors)
plt.axhline(best_thresh, color='green', linestyle='--', label=f'Threshold={best_thresh:.4f}')
plt.xticks(rotation=45)
plt.ylabel('Reconstruction Error')
plt.title(f'Snapshot #{anom_snapshot_idx}: Per-Node Errors')
legend_elements = [Patch(facecolor='steelblue', label='Normal'),
                   Patch(facecolor='red', label='Actually Anomalous')]
plt.legend(handles=legend_elements)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 17: Save Final Model & Download

# Save final model with threshold and edge_index
final_path = os.path.join(cfg.model_dir, 'gat.pt')
torch.save({
    'model_state': model.state_dict(),
    'model_params': {
        'n_features': cfg.n_features,
        'latent_dim': cfg.latent_dim,
        'hidden_1': cfg.hidden_1,
        'hidden_2': cfg.hidden_2,
        'heads_1': cfg.heads_1,
        'heads_2': cfg.heads_2,
        'dropout': cfg.dropout,
    },
    'threshold': best_thresh,
    'val_auroc': auroc,
    'val_ap': ap,
    'val_f1': best_f1,
}, final_path)
print(f"Final model saved to {final_path}")
print(f"  n_features={cfg.n_features}, latent_dim={cfg.latent_dim}")
print(f"  threshold={best_thresh:.6f}")
print(f"  edges={len(edge_index[0])}")
print(f"  Val AUROC={auroc:.4f}, AP={ap:.4f}, F1={best_f1:.4f}")

# Verify
print("\nVerifying saved model...")
verify = torch.load(final_path, map_location='cpu')
print(f"  model_params: {verify['model_params']}")
print(f"  threshold: {verify['threshold']:.6f}")
print(f"  val_auroc: {verify['val_auroc']:.4f}")
print(f"  edge_index len: {len(verify['edge_index'][0])}")

# Create zip for download
import zipfile
zip_path = os.path.join(cfg.model_dir, 'gat_model.zip')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write(final_path, 'gat.pt')
    zf.write(scaler_path, 'gat_scaler.pkl')
print(f"\nZip created: {zip_path} ({os.path.getsize(zip_path) / 1024 / 1024:.1f} MB)")

from IPython.display import FileLink
display(FileLink(zip_path))

In [ ]:
# Cell 18: Summary & Model Card
print("=" * 60)
print("GAT AUTOENCODER â€” TRAINING SUMMARY")
print("=" * 60)
print(f"\nDataset:")
print(f"  Train snapshots: {len(train_snapshots):,}")
print(f"  Val snapshots:   {len(val_snapshots):,}")
print(f"  Nodes per graph: {cfg.n_nodes}")
print(f"  Features per node: {cfg.n_features}")
print(f"  Edges: {len(edge_index[0])}")
print(f"\nFinal Hyperparameters (after Optuna tuning):")
for k, v in best_params.items():
    print(f"  {k}: {v}")
print(f"\nValidation Performance:")
print(f"  AUROC (per-node):       {auroc:.4f}")
print(f"  Average Precision:      {ap:.4f}")
print(f"  Best F1:                {best_f1:.4f}")
print(f"  Best Threshold:         {best_thresh:.6f}")
print(f"  Network AUROC:          {auroc_net:.4f}")
print(f"  Network AP:             {ap_net:.4f}")
print(f"\nModel File: gat.pt ({os.path.getsize(final_path) / 1024:.1f} KB)")
print(f"Scaler File: gat_scaler.pkl")
print("=" * 60)
print("\nHow to use in inference:")
print('''
  checkpoint = torch.load('gat.pt', map_location='cpu')
  model = GATAutoencoder(**checkpoint['model_params'])
  model.load_state_dict(checkpoint['model_state'])
  model.eval()
  
  # Normalize node_features using gat_scaler.pkl
  edge_index = checkpoint['edge_index']
  edge_tensor = torch.tensor(edge_index, dtype=torch.long)
  # err = model.reconstruction_error(x_tensor, edge_tensor)
  # per_node_scores = err.cpu().numpy()
  # per_node_anomaly = per_node_scores > checkpoint['threshold']
  ''')


In [ ]:
# Cell 19: Cleanup
print("Done. Ready to download from Output tab.")
print("Files to download:")
print(f"  1. {final_path}")
print(f"  2. {scaler_path}")
print(f"  3. {zip_path}")